# V84 → подбор параметров V93 → CSV


In [1]:
# %pip install numpy pandas catboost scipy scikit-learn openpyxl
import base64
import json
import zipfile
from dataclasses import asdict, dataclass, replace
from io import BytesIO, StringIO
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from scipy.linalg import solve
from scipy.optimize import minimize
from sklearn.preprocessing import SplineTransformer
from IPython.display import display
TARGET_YEAR = 2024
REGISTERED_FILE = Path('abitur_stage1/Зарегистрированы (2014-2023).xlsx')
SEARCH_SEED = 93107
CANDIDATES_PER_STREAM = 64
REFINEMENT_PER_STREAM = 24
REFINEMENT_ROUNDS = 1
COMBINATION_STARTS = 8
RIDGE_TO_ONE = 0.001
DEGREES = ('bachelor', 'master', 'background')
DEGREE_NAMES = {'bachelor': 'Бакалавриат и специалитет', 'master': 'Магистратура', 'background': 'СПО и остальные программы'}
SLOTS = ('none', 'first', 'regular', 'august', 'late')
pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 6)


## 1. V84


In [2]:
BUNDLE_B64 = (
'UEsDBBQAAAAIAPexXl1lUjyS4gEAAJ0GAAANAAAAbWFuaWZlc3QuanNvbp2U227bMAyGX6Xw9ZLZlmTLfZWhIGhHtgXIkqFD0DXI'
'u5dJM6DD1E3YnWmKH0nxpy7V5Oysl+r56VJ5tSSDXr9h1M7Sr/rIxLenCkNImzqBOqNJGOnrhD8D+Tt2rMnv1ay8spOCMDmv7oG8'
'5YxcEf2iIpydIQI5mobXt5DJuECcgDH5B+xH1dYtP9TdoW0rOvJhyUPLP1usqV6uZI8YlNH2xqy03VMM32fKPWGIcDsL564B5+Pq'
'FmfRwJzsSdvlOIVz9RGu70kv1Y5x/ZoiBoj8yH/F/dZgz6iSfwG6GuiKKVtUNkKIXuEWcriOdSU4BkFhcBbCirvKgJhs+gJQL2Ay'
'Cm2G0DMmSwgDhDTyOkOoB9YUEGSTixW85CIky/U+ZAs/Sw67184DIWgY0aeJZPwngA2iKSn8oYr+/2UhJKxo5r8IgvO+LQH1DwZM'
'aPToMdsYH7quRBWigy2ZqB9Imm/YkRZbjRmmpFmVMAUEve1GvbJTHtQX6U1wCG6OEFevwurMl7C2RD+CwZpG5QkCmKKzbnMpNwc5'
'FCmi/USBTUXMoTpRpI0aFo/2BMnqWWfHKYt2hEvwbkxk31SvbMg2ONQlDfL+vu09KLvQu5vjDKxEYlzApq3e8PUGFHW+InZ9ub4D'
'UEsDBBQAAAAIAPexXl1I7eUcjgEAAOEEAAAvAAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjYxX29ydGhvZ29uYWxfZnVuZGluZy5j'
'c3ZVlEtOZDEMReesxWXFv8RZDoJi2lI3+1ffgMDOMEcvvrZzqt5fP5/08efv8+313+eLDvXHmA8dZMGSBYRs84wCSoNHHe0+Ookk'
'xywSJBqsVmSSpHHLWKQ6eTWSpBE8VpFNYsaziI2WvB5DKNbkHUWUIiaLFDEQ5VxFnGJiYCsSFJo8s8hEM4NtF1l3dCLaeLegTYG1'
'jSorg0KiRwsaNuHKESVHLzWAGKlIrQ7Er2AJ3Jjs7cokN+XtRRa5ovtZJPGko9YNsvHqg6OC4IEEoqsyROjR0MCm9qXABAvn2Yr4'
'V9nZigQZ2tNq70sG4Vm7gw05WNoneSfDhOGc+ktggu62SBOCTv5TIY8bALsBhXGcUcAoeew690XnkcKcrc5QwqvpPEYcyVcBGCxs'
'7QraXuwVASOSfdZZrkjYgKF2VTw2GFsDfoDUVNDhzoQN4n3uI8N1I+/MfaZoizsa5AXkTCHVNTwQ5ahNQgP8lKsHvTcJA64jOoRm'
'WWDhdTlbQB7g7YvzT9BbwPNvXvUW9r3I/1BLAwQUAAAACAD3sV5dAXaD1JIBAADeBAAAIQAAAGlucHV0cy9mb3JlY2FzdF8yMDI0'
'X3Y1OV90NC40LmNzdlWTS24dQQhF51kLDxXfopZjOc7UkuP9K2ApDTXs87q5fM77/fb9AX8+vz7e3/5+/+LF+lr+4gXiKKcBgS5k'
'bcCwcPWj3I8KxAuPNbEkimdUcKAwJGqygXmje5MANsLx0QESQn6ArBG8X4vA9kLbTRgsS1gDSWAYgyiYO0Y0MTDhJzeBZyMH1yD7'
'Dg6wyJhR9YDpRjoPoQVWO5Am9JNjXZYY1KU3l0SASa46ekWTgebMMYo4aLXfA9EGZcHwJgFyCPd454CoI/fuUgMyRulm0oMZnRaI'
'HzzdW4ogFuijiCaRcYBUQURRqUmpQBjjow07cPem0oQrOD1YaWjPkyKkGMQNCNjx/J8vyo0ELg04fUMdQCC6RJQYHRplhUg3FT9W'
'jEGipCjQz+nvamuipCB/dhGlxJ49El2JaQMf5PG7VMvPdqNcSBDdc9qQkWOolIEU93gjXUAZNePOPDVDz1QSZI/egGqGPQDnfxt1'
'AMmpVifyvccU4HrMBqn9jjo+rXmoPH6Csfg6P7fNUdc/7XLU8SviH1BLAwQUAAAACAD3sV5d4EkLw5ABAADfBAAALwAAAGlucHV0'
'cy9mb3JlY2FzdF8yMDI0X3Y2MF9jb25zaXN0ZW50X3N0cmVhbXMuY3N2VZPLTgYxCEb3Pgs/KXf6OMbL1kR9/0jHKMxyTlo+Cmde'
'n7/f4P3j8+3l+ev7iRfrY/mDF4ijcwMCXZjZgGHh6k+5fyoQLzRtYkCyUKOJA6WjNAjgc4SaJLAT+iizq4xjNyZrBMdjEVhsTG7C'
'YF6tDCJg5pjRROvWwt3AwITRrIkDa6Bkk7gnJ1gGaoNdMYzRN2iBsaFIk2pXCYWaMGg46jgjwGujjTN6SyYDzZpbN0sOWmXZmwQo'
'87yUIHvNF9IGMcPV/ZYFZIrRZ0qDmVwSiDuuHlx5IPXo0X6pIOpIgxhI7dl6/pcKgjbqBNQaV7+5VLgllwe0ZnKJUEWl5y8EHPg3'
'gjxqcP7vOI8YrOjRQCBwjwtzzHmcEGuP8lKiVNQGcRJpROSJsBGxa5ftcx4lcvRIdIssGaTssAblgiFLA70SuqmSgWU2VS5QyT6u'
'lArIo2beM/d51b+ReVmw56tKAsr2JI8GJD36PBbEnCzfJ1kC3D6rQ0YfAXHATMzadv8Bea1fkRrU9ksH70nK7yR/AFBLAwQUAAAA'
'CAD3sV5djm73vpABAADkBAAAKQAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y2M19zZWFzb25fc2hhcGUuY3N2XZNNbp4hDIT3OYs/'
'y3+AOU7UpttKbe6vDIr02mTpR8AMw/D7/fOD/vz99/Hr/f/nm4nFS+bLhDw5rYBSCOcuYCQsNfo9BqkapxYZpLZ4ZZFJuoJ3E1lk'
'ZpdsksXiEUU2TlaW+RCXJr1eojTm5L2KGI1Q9kYcRDhGkQAx9igyaMgswyCTzMe1a93SSWMZr7Zg0zhB6kNUaOjirDUKw+osVsQo'
'YrKWGXXSrRztnLikdVDMzdIWTAqXnoMuCm1hgiQ5DHczm9yRVZlBFTRgr9agC10aTfApbHUsyuC+WduWAPEeL/rgiDfL3unDNN4F'
'FoLpF0AbLmFUQTYvfwiqAN1dIbgSXuy5X552WFzA0Lm6XZ5u4NVnzT3oPLVAPjMLzHNirALrgDbnUbC2A76Rjj0AnVgsUbNekqiD'
'7T47esjhBeIoZEmiDSa8doGTLUt5OF2otuRpwiW5f3g+LcCnqlhQAgCtE1AD/PhRrtCCZCsPdgeJAlwjHCpnE1h4Xc5mIc+KFtt5'
'f++XwPNjy6gc/DvIL1BLAwQUAAAACAD3sV5dNG3uS44BAADyBAAAIgAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y3NV9jbGVhbi5j'
'c3Zdk0tuHUEIRedZC0b8C5ZjJc40UuL9K5SVCPoN66iKC/TpH++fH/Dz1++P7+9/Pr8Jib1RvAmBKmoMYFDBkwMECInmrC9ngzLU'
'BRyYBGOVCGAlVB9yQJTx8JAECUNZrwoqMQco7eTzRgxeK6iJgHuh1RAFD8ewIdak8MgQBxdDOkOi2zP0GHJeshPCFHPdKPDj6FOF'
'qQmjzyvujrVmyCYCdgpz+mMFIUJexJ7Z3P2SYUw2B5gGEg85YByoq0qCUWCu/grUHlO2Ddz9pQ7hZ3bboKFYM0H7oK2QTVIboZIj'
'1blKqPTOV91Woue2Vbk7NswF8iW6gNmRh+iXvGt3yiAH6/+IeRW5IAYI9KeWdaMFwaA5P5adV47uXGpA3Aq6KpwLfGXkVwYPKJD2'
'YECb0RvzOfMztK2QwFgXWgpFnox2ogHpAL8ZazC+G37U6D5RVki+hHaXtHd1bfDRLK8M9wPYAAEWtHVD+9+fL5/XhUdIm/A8x31Q'
'02VrULvrtqAvbFA3c83RFjCjznb13zb/AlBLAwQUAAAACAD3sV5df8AJDBoCAADQBQAAIgAAAGlucHV0cy9mb3JlY2FzdF8yMDI0'
'X3Y3OV9zdWI0MC5jc3ZVlEmuGDcMRPc+Cz/BmdRxDA/bAInvj1Tb8Ke6dypIVLH41N+//vpBP//598e3r//9+mJi8SH1YUIhPKdS'
'akWlVE6dnFzRSFh26e9lkJazjLjOqkk6wqWnzFct7BVWwXdWbTJrrkm9nAxZJrt75lXgkGpyKxxuAZfLUX+IUlVzj830qkblyQdf'
'6qpOeZwz2o+tGpTZrClW196kOMZeWjOrFpnr477LV+23paHKYlU/elY9lHEQd87Za1RgdNjc+jMPqJhMBmvMia2gRmHFFdqVqzrC'
'T66uc4nxcqRJKcFhcTvSokBIM5Z23d3k5yCOUN2QdABQ8YjMFagecocjbev4VAEbTnN3SG9d0HZbAmeB4Z7BdNcScPMyhGTtew2o'
'88Rh9/arYGLvcEnUrNGHuomHRZOdG6hTR6DWk5fReVsCb1N8zGUdATfgGhOSW86VQIGgbfu7cx4SMZ0slbxEI8POMc/VnAqsK1hc'
'7R7YPPQ9+JbUqRUBX3IEsLjEfm4NhGC6It6TACqRvqygvWQzd7FPEfwpfgypWmtQ9WUG3KHbzMrYe4EdGktgdxULMoQsJXZtzMe1'
'6+m8bqjfXkbs83HNb/IY/yLXNa3ztnLIhsdNNjrgBieNsWmsqKSHDQ9o6wM5dTCAh75O7BkGcD2+lu09DID2WhYdPmiye7X+E6O/'
'Ss8jlmZduT2QOXf6uY4DMvwZQiS3A/8zhP8BUEsDBBQAAAAIAPexXl3EHA+6FAIAANMFAAAcAAAAaW5wdXRzL2ZvcmVjYXN0XzIw'
'MjRfdjgxLmNzdlWUS85VQQiE566Fn/BsYDnGx9RE3X+so4n0md3zpS9dBUV//fz7G33/8fPbl8+/fn8ysfiQ82FC3nzkWNdCpTgc'
'1u250EhY9tPfn0F6lLVlXJcmaQa7hJYvPTh7uNxO29IiC+VJ/OM626DNI9keS4fUgiVVZWW7XJLqQ5RONVsfXL/U6Ehzt9mcpU55'
'BgWt86JBGcN+cI8tTUoZrnHRXHrIUjhbz1y31VtSU45z+4TO0qFMxxC00v9TFTpekG8TF1VIKq7suISqUbhzKlytUHXCTzZLl6tC'
'vCRpUoyxYBZTSw8FJEmPVC8tCiv2jlu+NjlMzTO3S9KQh3Mc7bN1ETfNgVCMLpbqSxKS5lOQ1FrbSgTO0/iMyOzlyJ17cHjWuc4m'
'aHKL6yX0yV0LD+bvFy1SCTapuqwidy9JCFwjSzp+TReBc4HF6C3nSgaDbvXfdj9RtGY96PIyIzOGQyR7odPhFCubZffA+okfYuEa'
'2rnwSR+y0hbXDfUXWh7ThU06HDmWvRD2sKaVu6b9BFAFolsv1aovMQieKxc60NcZf5wFHpW5qgUZEup6bFk+WqZaJxZiSo7HYUov'
'iORxSLRdFfstZZ4eW5bqukXeTBh7ZLMNRdy0OcOwcAsNTwpEw8qyZxpRcfXU3sNA0F6f52lZZE6u9r8BQ4+s6tKAMShLJRZh4TzW'
'sa2n9yRChicoT1w98n9T+ANQSwMEFAAAAAgA97FeXQSlxKgYAgAA1gUAABwAAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92ODMuY3N2'
'VZRLjhUxDEXnrCVtxX97OS1opkjQ+xc3rxFODeson+v4uH68f36sn79+f3x///P5TbbY24432cs2VXikDuRlSp3MIQNlbdrzqc9P'
'W6xOzNZxHeQvWhksF40lO6nKpH1oLtEmjW63obXEk7LTq4Y2zt0Uvr0moe4rUr5tXh5F7HtXDJXlVuSWYjZUX2s121qH2vJW2hzq'
'NdSxFhdtbuWhKCoRlKM6h+YzUq1IIQmpvg7sFSLkrFVDeS8vpdiWeyLxKaopLSSnKJZljYsCl08k1sWNSIoXlaH2iMSohk81qXJt'
'DRigJHj2vGguYxwYLXJFqoV3ozJlv2gvNSE8Bnf/p9CNPUhUZftQfkSCadpO6E/atUiXwoTKFpvK4d25pjmzr7UOysRdfQWFdxCa'
'UALA0Fx+FM2o6Ru0eySCb5Dm9KLnbvimaGYKMg3kJYUCC4v/wToqKuI4RNKBAmMgV+7igbqSUHdf6+6G1dEPIwWB1GNgvM4yjKwP'
'xEQFWbmZDDwDRYGB0h7YS5ii1Hu2Q8Agd5aa3cyPLPAOY+joiEwJ0E6UEv8UvzbaEoOKATzQDzTMgtbA83egM1x9QZhHvT1rHpXr'
'meWrBjQ8prCXb9RSmIyBDA/Qt/QJDecYI9hbdw7EAG0SdI2vI5/tgGqPTzimxPizXS8CxQD3dolpGRRjo114k3mmIxpkxB/renR4'
'hu3evK+W6Vcn/gJQSwMEFAAAAAgA97FeXWU5onAaAgAA0wUAACIAAABpbnB1dHMvdjg0X3ByaW9yX3JlY29uc3RydWN0ZWQuY3N2'
'VZRLjlUxDETnrCVt+Rd/ltOCZooEvX9ReYh27jClxCm7TvLj/fNj/fz1++P7+5/Pb8rqbxxvysuZxHenjyhrM6mzmIyoi4lnac+l'
'L9lJsplzxL0kgjxQ/KoeUDdtdi4bNZeakzL3fWkt3UIVKNGj9hIpEhG3UY0vR/nGssKVssRaR9UV6pRhFTmqrV1OYsoSo/ra2ync'
'2EbcyzsodkRdZWOpJ204+poj1Hw6qhXs5JqsV8F+3eJeqvKlCq9ABpIpOY4EwcC9l+x9qbrcglIrrwK2pJWEs+Pa6g9HgmYK42W3'
'6lED9YostOw6mssaKlK61QI/mFyb5lWhl2Eczap7hgTWZCv5NpMxCthuS8DMxWlH89UNaENgpBWyZ76AzkCvWdoVJqgzh9EsRD8q'
'EmK0ioh6jII6gf1y9DRNgbqHJeB26HDdF0nAzeTVTF6iLFMKibb/NuuQqE3W8G4jKninfYCrEW0Ba1bu6/QdWR3+TjZ4JSwjorkg'
'7vIvWOrQp0kRWfdOPCg87Wzl61o0CM7Nwuc4CGzKtrY9mjy8ADw8WcPn4TmincaEd9l0C+4gdmVexfZSAFrKoSPGsSLqu8bfQY96'
'b8lrYz2tgDjG08JPM1YAHG5gLem5FrxJI7fsa1agTpQ6EjCPaAvIJd7sdfqZBlh7LOGe8bnAxvR+GBN8dy18Wauz009g14WvHOTw'
'PKKdHIKZa1zYvxz+AlBLAwQUAAAACAD3sV5dw/6xWpEBAADjBAAAIgAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1OV90NC40Ny5j'
'c3ZVk0tuniEMReddi2PhF5jlRGk6rdRm/+rlj6rLN+QI/Dz8fP/6lF+//3x+vP/9+uHD823MNx8SUz0JTHKoXcBl6OAxnscU86GL'
'oABSZ5FMsU7tTbLEfWlcpMXL1I1ki4Wp8U6MK/N6Gya1kDlJXApBcpMESGkWSUrNqXm9Kikk2otkopjWNUnWM3VL9dBwki2VpZvE'
'hpSHNsOafScKEpecodEkIW7BrkHykdoKT/Zdm03JcB1GsiSRuq6wLbFNF+dgWyKxeBYDE6wQh6+gwp0aIgRSB4PAhcCgitVCh6jQ'
'eYUtiUjlvF86oJbrypLV96RgwyMxVBgYLnuGCpjk1XKY+NT+X3sfOwAqCBzOsfg+brTadeEedB8tAo1sAliR3HkfKQ64IrTY5n77'
'WGGT6+0jxdLNmGaPnPDBNz9jv3RwnTznOa8LlOD3xSLAdJPz75cMGteLfqbcpwljV0cD1Mg5wQI0sRgRHphrGkHIfLx4ThIGPI6o'
'EP/6er6wXi32gPUDTNZ8BHBtFon977uk+J7jP1BLAwQUAAAACAD3sV5d1ntjLYoBAADkBAAAKQAAAGlucHV0cy9mb3JlY2FzdF8y'
'MDI0X3Y1OF9oYWxmX3N0cmVhbXMuY3N2VZNLbgUgCEXnXQuPCIjgcpp+pk3a7j+9tknBoSfgRT2+Pn+/0fvH59vL89f3kw6dj7Ee'
'OmhOXl5AaDpHA0qDRy3tXk4SMw4t4iQTJVlkkezFEkWCFDW7QJKuwdliNzZWHlVjoyXHYwj5Up5eRMld2aKIkYfzyiKTPJPDiji6'
'jLcUWaRhPGaRuKOTFm5Nd5FNS5S9omUgerNWtPwNvBtRmhsDV5AYqRhLq5lXtGBaxV2uIgsPtng0EjQtL5JkG69UYJPNZKsSiCDL'
'edeZYEJPhgdzKGvdN1QwHKh2hQw2g70uFzKYOUud8MiAB2jPBhlCr5K8g2GCXrvCBFsc9WImZG20PHKYcKwCCsFK6zxqBKfUut9z'
'HivMr3pIkZxaAA4Hr1kADjtny9ykTaM8TgRrZYpcmdABQ88oYGfo3jAPkAb8RHgDGBN71tyQwXnWUJJ35u+MYv/geJD9FNAAwLKA'
'4oPzai1GIvXt8njQMyDBtYQBGKn1x9lwtP48Fa1jn4I2EwTAL2kd9neTP1BLAwQUAAAACAD3sV5d6+03jpcBAADgBAAALgAAAGlu'
'cHV0cy9mb3JlY2FzdF8yMDI0X3Y1N19zdHJlYW1fY2FsaWJyYXRlZC5jc3ZVlEluJDEMBO/zFjbBXdRzDC9XA7b/D1MzmCZ1VHSV'
'kkpF9dvLzzt8fH69v758//wREntQPITABcMaMMTClQ0ECKmXei8NWBbabuLAbLi9SQDvxIwmCyQCacQmyEokbrKBVdGbKI3o9aCa'
'VOoBbyLgTijSRCFIkAaxIhuTmzh4Ju7VJEBNkHeTdUcnLDHUscmGUB+PcBW7HHc/wgweVUMDAUufw7GC1HWoNrErmGtWI+SxSYAJ'
'dXFFFhgpZvfCCboNZWy7QRfNE5YIQoR7EL6iywPThdnTlgqagmZNDDSKNHBQT4zu9q8Mhj1uuVDrHQ3yzj0e7PZpHQ9sVq1ch0H9'
'P3keM3RcaB4x1NryPF6wtOV5tOjQPE6UAOPnMsKRtcE64ClnHiWUMQbY5TiaPEEZsXD3DMxX5LGh+u+1ng2fH00eFSry2UMeGeoD'
'GTtEfUN9p3lMWOhjh7wT90nwPuVxwDF65FKgzpBdZEnAOYYsBVhnsXL3WPd/LevyF9JIXMBxjVD/A4w0EvcB3qeu269XuIvWfz3+'
'AlBLAwQUAAAACAD3sV5dS33DyCYBAAAaBAAANAAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1Nl9tdWx0aXN0cmVhbV9zdWJzcGFj'
'ZV9lYi5jc3ZVkztuAzEMRPuchQbEj0TqOEbstAES3x+hiuyMyweJHIpv93F/PeXr++f5ef99fdiwuI11syERIBXfIJMBcIYQGxM4'
'RbcDl6gvYPYpZZSYUciWzIt8XCl5GypLJ9Bk5gK6zDBgSGzCKZEbuMQ8gckxJXNR5ZY16kIdPR+h9hCYSa1jMJO6aNHloBidMjdG'
'0NWNMKBmj099q7VgJt3i0y9saWoBVIppab4Q09rcFdjikhq1uELMEVd4zRGndLk4ZrdjYHtbeLerxH9dHY3X11DHotOZCx1hXXUE'
'WoLa3wTl21m90eabxx5AqX+bo6oWx1W9JQP1JAE6XxIohaC4/+Z7bYxJxRRkvXVQf0JIM15J2yJYb1XnHwOVKNZ6VOFtbUoxiZ+V'
'/AFQSwMEFAAAAAgA97FeXRe4zZQlAQAAGQQAACkAAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NTVfc2ltcGxleDNkX2ViLmNzdlWT'
'S27EMAxD9z0LA1iSP/JxBu10W6Cd+6NMgYbK8kGWRPslH4/XE59f38/3x8/rzZv3o83DGyJFhtgiRxNEhQ5bQzjgXnCyWoYuVk2Y'
'8BnCDTO/MNq1Zh3NMK41RMcYXRismrCz6sKB0VM44bscXnVNYrZS25imQdaQoaox01YIY6amiBZw02TrZY0x0ZjCieFbuNB3OZzo'
'pvi2EUtVWrMeQitraC2mOuktutbQXLjynuZCmf7M6a40t8qWrFs29rqI1rreJPgV/bflKfF6vjwdVgosgR4rT32eItobIn5QpS1v'
'tCvR3RRYmU9v9VzU+bTmyk9t3kXzRgulLet85hDRVyWDbZHDNIS6TLu9PgldFZi3rkVvooTp2ucPJqKp0hfnk/wCUEsDBBQAAAAI'
'APexXl3sT1ohIgEAABkEAAAuAAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjU0X3NvZnRfdGhyZXNob2xkX2ViLmNzdlWSS04EMQxE'
'95ylWoo/cZzjjGDYIsHcX7iR6HIvn+K4HL98PF5PfH59P98fP683HerHiEMHbJMEliTFIFgHhywlTqgaMSDZiledtuKExiRuSDu1'
'ccWsYwhmJFELJ9EwlxEdc27ixPSGARtCXD0mEdcIhRshfqEMrGSq1EwZRIVvjihWb2WMeIuRmigYI4Epi7jg2U4TPlrqhk2+tazJ'
'ZOfS1kDLKG+WN/OGDhOu5c8cU09zwfHLXFtwiWspG8lnl7WWYfWL/ilPiWakmk5IhkXgsvLU1+sC2grrQyUpb7R7ZbkLgrT+5U03'
'qbS1Qody/tKmToobLbTC7P1rDr2ofCnPSpcwu3wJm5Qu4ba0r6RcNYjbrVXeSHmj3clGv2fnSn4BUEsDBBQAAAAIAPexXl34sTXa'
'JwEAABkEAAAwAAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjUzX2h1YmVyX2ViX2F1dG9ub21vdXMuY3N2VZNJTgNBEATvvCUtdS29'
'PccCc0UC/19UIzGZPoZq7YqZj/vzgc+v78f7/ef55s3z1sbNG7KRDDFJDgmFQsI0scOyEwdsBXHC3YkLPiS6q9W4MNo1Zt6aoUcn'
'OvpVWRiFScxX7Og5iQPhizh1zMLog7gxGiutoU8n1k5doo6crLWAJze2lDFWG3UjjqqURhOZXNAWYghuhPFxZc3kraWNY8paBq9U'
'3iIkmojGJcqcN4455lKiE1OmLJ2yIUcpa3LrqK/o/yTrSIxF8vJACkgij7WOvusDW3/2jDRfqC4ldRu+Lyp3bGIm/cub9DjaJDG1'
'Y2lzLlnWPEgTnbC0/9a844s3KF3OzPJlbFK6TOr0JOVKYLxUTVxfxzqijK85PxipTElmnJP8AlBLAwQUAAAACAD3sV5d7/RLHCIB'
'AAAZBAAALAAAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1Ml9hdXRvbm9tb3VzX21ldGEuY3N2VZMxbsQwDAT7e8saEEmJkp5zSC5t'
'gOT+j9ApvOtyIHJJa+TP5/uFr++f18fz9/3w5v1oeXhDN5IhkuRohFDosOnEAY8gZp1u4oSb4IKn9G5YZ3K0a8w8mmF0JzrGCmJg'
'TCnuGCnFo3o7MREhxVPHLGRK7kYag6xhGoOsdsokOrosYQHvi9hljNVGMsYSfU3iRB8ydZUJCdoIY1RZM1mitMlZbTSYW97CBTt8'
'D+KALY45zYUUT0wuXOJkytajstbZFobL2jolxiZ5rUMKTAIva536tCtvXfNGdVPSt+HronKXBJP88na92PWvTbq6JpY2l5Csp06a'
'GISl+bUHj05f/JrS5UZymFRGGSDplZQrgbx1zRut+v9IW6lMXY95naba4w9QSwMEFAAAAAgA97FeXaqnmQ0nAQAAGQQAACoAAABp'
'bnB1dHMvZm9yZWNhc3RfMjAyNF92NTBfZ3JhbmRfdW5pZmllZC5jc3ZVk0luw0AMBO95SxsYLrM9x4ida4DE/0dah0zTx4K4tFjS'
'4/564uv75/l5/319ePO8tXHzhmwiQ0yRozyKCgmLEHbiEA7YTuGEexcueC9LNsz9YLSzZt6aoW8TOnobwkBPFyb67MLO3oIDvkrx'
'rGsWxnkb4sZo+6A19KFOMwxbQmay8jTgJZNlWWNM1LXGBnIpoE1imbsQ54bEjejKRGs2NJnaCjgiNYjeomkNzUUoIM15lrk0l7ow'
'zU1loLiyZWOqkNZShcGv6N/auiRWcvgWBYZAx1qXvtpFe1PED6q0rbeJm54P0Z3azMp8eqt1UefTWplPbZWYpIsmCqw6nzn80OVL'
'26jLlJi+TPOpy0JUT0JXBcZb14QpCEWZ3ub6wVRJU5ai6yR/UEsDBBQAAAAIAPexXl2yeIPpIQEAABkEAAAtAAAAaW5wdXRzL2Zv'
'cmVjYXN0XzIwMjRfdjQ4X3JvYnVzdF9jb25zZW5zdXMuY3N2VZNJbgMxDATveUsbEBdR1HOM2LkGSPx/hDrEzTkWxL1mHvfXE1/f'
'P8/P++/rQ4f6bcRNB2yTBBYkxSBYB4eoEidEWl5AMokLqpOY0Gi5u3IZbOPdZt2GYK4gKnwb0TBNiF7Bkzgxs+UGbChx9TaJkBa6'
'C/kq49JGaqbNNqKYylyxWm4TvbWRmsgbBjwbrsIkJny01w1bHL+sSbBraWNoWTN1otXirFvmTBtO6OS8x9xsbRaiVcreZaNdt6w5'
'q1h9Rf8HyyPRBkkvZAgCj5VHn25SQFtgfVCTlJfIjfd6edwxTaTVL2+t4tG2SN7rl7YeGZe3hQbZ69ccvMHxxalKl3Di8iWsX7rE'
'Sf0k5apBXLJWeSMlhNucH4yRZarVt3OSP1BLAwQUAAAACAD3sV5d/YUpISIBAAAZBAAAKQAAAGlucHV0cy9mb3JlY2FzdF8yMDI0'
'X3Y0N19zdWI0N19lbmdpbmUuY3N2VZJBbsRACATveQuWBpgB5jmrZHONlOz/lXakuPGxhN20KX88Xk/5/Pp+vj9+Xm82bB4jDhvi'
'm6TiQTIZBO8wRbWIC7iIIVqTmGLWdpRYJHGLDqKPa00eQ2XFJpqsEUSXNZ04ZVXDBSxiiI8WlX1NSdgkbiCnOmR5EtEpWULRybhG'
'XSwZpbOtUTRyI4bMatOUmW1NwUvL3eLJh2FNgyWgrc1M3HkHeHPlh8PcDRdksO9pLlpUSiqp+pYt7diwNhnq+Iv+M+uU6INkN8K5'
'CDxWnfr6c4GzkvBDBQn/0yZtuXTU6Y4j1ZYPby3jTxtpii3SuhGa8NsgrY2q56OHXnT6YivoUtaCL2UT6FLmWz8JXDWI21spyiIQ'
'1We7E0y1fD9P8gtQSwMEFAAAAAgA97FeXUPTj3MmAQAAGQQAACoAAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NDVfbWluaW1heF9z'
'dWI1MC5jc3ZVk0tOBDEMBfec5Y0U27HjHAfBsEWCub9wFvRzL0vxv7o/319PfH3/PD/ef19vOnQ+Rjx0wDZJYEFSDIJ1mBBVokOk'
'5QUkW9EFVScmNFrurty80MbVZj2GwNckKuY2osFNiLOCnejwbBiwocTV2yRCFnEjxr5Qxq2N1EybM4nCNYhWyyVxtjZSE1mrG5ir'
'4cJMDiFZXtrrhi0OUdYkuFxpY5uyZtrerBbnRGXOtKFDnW2OOeeVylzw3iWuddlIrl3WJqtYfUX/s+aRaIOkNzIEgcfKo083KaAt'
'sD4oJ+UtcuNaL487pom0+uWtVTzaFmn2+qWtR8btbaFB9vo1B29wfHGq0iWcuHwJ65cumaR+knLVIG5Zq7yREsJtzg/GyDLV6ts5'
'yR9QSwECFAMUAAAACAD3sV5dZVI8kuIBAACdBgAADQAAAAAAAAAAAAAAgAEAAAAAbWFuaWZlc3QuanNvblBLAQIUAxQAAAAIAPex'
'Xl1I7eUcjgEAAOEEAAAvAAAAAAAAAAAAAACAAQ0CAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NjFfb3J0aG9nb25hbF9mdW5kaW5n'
'LmNzdlBLAQIUAxQAAAAIAPexXl0BdoPUkgEAAN4EAAAhAAAAAAAAAAAAAACAAegDAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NTlf'
'dDQuNC5jc3ZQSwECFAMUAAAACAD3sV5d4EkLw5ABAADfBAAALwAAAAAAAAAAAAAAgAG5BQAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRf'
'djYwX2NvbnNpc3RlbnRfc3RyZWFtcy5jc3ZQSwECFAMUAAAACAD3sV5djm73vpABAADkBAAAKQAAAAAAAAAAAAAAgAGWBwAAaW5w'
'dXRzL2ZvcmVjYXN0XzIwMjRfdjYzX3NlYXNvbl9zaGFwZS5jc3ZQSwECFAMUAAAACAD3sV5dNG3uS44BAADyBAAAIgAAAAAAAAAA'
'AAAAgAFtCQAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjc1X2NsZWFuLmNzdlBLAQIUAxQAAAAIAPexXl1/wAkMGgIAANAFAAAiAAAA'
'AAAAAAAAAACAATsLAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92Nzlfc3ViNDAuY3N2UEsBAhQDFAAAAAgA97FeXcQcD7oUAgAA0wUA'
'ABwAAAAAAAAAAAAAAIABlQ0AAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y4MS5jc3ZQSwECFAMUAAAACAD3sV5dBKXEqBgCAADWBQAA'
'HAAAAAAAAAAAAAAAgAHjDwAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjgzLmNzdlBLAQIUAxQAAAAIAPexXl1lOaJwGgIAANMFAAAi'
'AAAAAAAAAAAAAACAATUSAABpbnB1dHMvdjg0X3ByaW9yX3JlY29uc3RydWN0ZWQuY3N2UEsBAhQDFAAAAAgA97FeXcP+sVqRAQAA'
'4wQAACIAAAAAAAAAAAAAAIABjxQAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1OV90NC40Ny5jc3ZQSwECFAMUAAAACAD3sV5d1ntj'
'LYoBAADkBAAAKQAAAAAAAAAAAAAAgAFgFgAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjU4X2hhbGZfc3RyZWFtcy5jc3ZQSwECFAMU'
'AAAACAD3sV5d6+03jpcBAADgBAAALgAAAAAAAAAAAAAAgAExGAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjU3X3N0cmVhbV9jYWxp'
'YnJhdGVkLmNzdlBLAQIUAxQAAAAIAPexXl1LfcPIJgEAABoEAAA0AAAAAAAAAAAAAACAARQaAABpbnB1dHMvZm9yZWNhc3RfMjAy'
'NF92NTZfbXVsdGlzdHJlYW1fc3Vic3BhY2VfZWIuY3N2UEsBAhQDFAAAAAgA97FeXRe4zZQlAQAAGQQAACkAAAAAAAAAAAAAAIAB'
'jBsAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1NV9zaW1wbGV4M2RfZWIuY3N2UEsBAhQDFAAAAAgA97FeXexPWiEiAQAAGQQAAC4A'
'AAAAAAAAAAAAAIAB+BwAAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1NF9zb2Z0X3RocmVzaG9sZF9lYi5jc3ZQSwECFAMUAAAACAD3'
'sV5d+LE12icBAAAZBAAAMAAAAAAAAAAAAAAAgAFmHgAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjUzX2h1YmVyX2ViX2F1dG9ub21v'
'dXMuY3N2UEsBAhQDFAAAAAgA97FeXe/0SxwiAQAAGQQAACwAAAAAAAAAAAAAAIAB2x8AAGlucHV0cy9mb3JlY2FzdF8yMDI0X3Y1'
'Ml9hdXRvbm9tb3VzX21ldGEuY3N2UEsBAhQDFAAAAAgA97FeXaqnmQ0nAQAAGQQAACoAAAAAAAAAAAAAAIABRyEAAGlucHV0cy9m'
'b3JlY2FzdF8yMDI0X3Y1MF9ncmFuZF91bmlmaWVkLmNzdlBLAQIUAxQAAAAIAPexXl2yeIPpIQEAABkEAAAtAAAAAAAAAAAAAACA'
'AbYiAABpbnB1dHMvZm9yZWNhc3RfMjAyNF92NDhfcm9idXN0X2NvbnNlbnN1cy5jc3ZQSwECFAMUAAAACAD3sV5d/YUpISIBAAAZ'
'BAAAKQAAAAAAAAAAAAAAgAEiJAAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjQ3X3N1YjQ3X2VuZ2luZS5jc3ZQSwECFAMUAAAACAD3'
'sV5dQ9OPcyYBAAAZBAAAKgAAAAAAAAAAAAAAgAGLJQAAaW5wdXRzL2ZvcmVjYXN0XzIwMjRfdjQ1X21pbmltYXhfc3ViNTAuY3N2'
'UEsFBgAAAAAWABYAUQcAAPkmAAAAAA=='
)
archive = zipfile.ZipFile(BytesIO(base64.b64decode(BUNDLE_B64)))
manifest = json.loads(archive.read("manifest.json"))


In [3]:
def read_forecast(path):
    return pd.read_csv(BytesIO(archive.read(path)), parse_dates=["date"])

base = read_forecast(manifest["baseline"])
config = manifest["config"]
base_log = np.log1p(base.forecast.to_numpy())
active = base_log > 0
rows, rhs = [], []
for entry in manifest["basis"]:
    delta = np.log1p(read_forecast(entry["path"]).forecast.to_numpy()) - base_log
    rows.append(delta[active])
    rhs.append(0.5 * (np.sum(delta[active] ** 2)
        - config["assumed_evaluated_days"] * (entry["score"] ** 2 - config["reference_score"] ** 2)))
D = np.ascontiguousarray(rows, dtype=np.float64)
v84_weights = np.linalg.solve(np.dot(D, D.T) + config["regularization"] * np.eye(len(rhs)), np.asarray(rhs))
adjusted = base_log.copy()
adjusted[active] += np.dot(D.T, v84_weights)
closed = ((base.date.dt.dayofweek == 6)
          | base.date.dt.strftime("%Y-%m-%d").isin(config["closed_saturdays"])).to_numpy()
adjusted[closed] = 0
adjusted = np.maximum(adjusted, 0)
counts = np.expm1(adjusted[~closed])
values = np.zeros(len(base))
values[~closed] = np.round(counts * config["target_volume"] / counts.sum(), 6)
v84_reference = pd.DataFrame({"date": base.date, "forecast": values})


## 2. Исторические данные


In [4]:
raw = pd.read_excel(REGISTERED_FILE, sheet_name="Sheet1", usecols=["reg_ts", "zayav_type_name"], engine="openpyxl")


In [5]:
def campaign(year):
    return pd.date_range(f"{year}-06-20", f"{year}-08-31", name="date")

In [6]:
registrations = raw.copy()
registrations['date'] = pd.to_datetime(registrations['reg_ts'], errors='raise').dt.normalize()
registrations['degree'] = np.select([registrations['zayav_type_name'].str.startswith(prefix) for prefix in ('Бакалавр', 'Магистр')], DEGREES[:2], default='background')
source_years = sorted(registrations['date'].dt.year.unique())
date_grid = pd.DatetimeIndex(np.concatenate([campaign(year).values for year in source_years]), name='date')
daily_counts = registrations.groupby(['date', 'degree']).size().unstack(fill_value=0).reindex(index=date_grid, columns=DEGREES, fill_value=0).astype(float)


In [7]:
EGE_DATES = """year,russian,math,social,english
2014,2014-05-29,2014-06-05,2014-06-11,2014-06-02
2015,2015-05-28,2015-06-04,2015-06-08,2015-06-11
2016,2016-05-30,2016-06-06,2016-06-08,2016-06-14
2017,2017-06-09,2017-06-02,2017-06-05,2017-06-13
2018,2018-06-06,2018-06-01,2018-06-14,2018-06-18
2019,2019-06-03,2019-05-29,2019-06-10,2019-06-05
2020,2020-07-06,2020-07-10,2020-07-16,2020-07-20
2021,2021-06-03,2021-06-07,2021-06-15,2021-06-18
2022,2022-05-30,2022-06-02,2022-06-09,2022-06-14
2023,2023-05-29,2023-06-01,2023-06-08,2023-06-13
2024,2024-05-28,2024-05-31,2024-06-04,2024-06-11"""
exam_source = pd.read_csv(StringIO(EGE_DATES))
exam_calendar = exam_source.set_index("year")[["russian", "math", "social", "english"]].copy()
for subject in exam_calendar:
    exam_calendar[subject] = pd.to_datetime(exam_calendar[subject], errors="raise")



## 3. Модели и параметры поиска


In [8]:
@dataclass(frozen=True)
class DegreeConfig:
    family: str = 'catboost'
    coordinates: str = 'event'
    depth: int = 4
    iterations: int = 700
    half_life: float = 3.0
    align_history: bool = True
    level_method: str = 'none'
    level_window: int = 3
    exams: bool = False
    knots: int = 9
    penalty: float = 3.0
    history_window: int = 0
    learning_rate: float = 0.035
    l2_leaf_reg: float = 10.0
    positive_only: bool = False
    loss: str = 'RMSE'


In [9]:
def saturday_slots(dates):
    dates = pd.DatetimeIndex(dates)
    position = (dates - pd.to_datetime(dates.year.astype(str) + '-06-20')).days
    return np.select([dates.dayofweek != 5, position < 7, (dates.month == 8) & (dates.day > 21), dates.month == 8], ['none', 'first', 'late', 'august'], default='regular')

def nominal_deadline(year, degree):
    if degree == 'bachelor':
        return pd.Timestamp(f'{year}-07-29' if year == 2021 else f'{year}-07-25')
    if degree == 'master':
        date = pd.Timestamp(f'{year}-07-29')
        return date - pd.Timedelta(days=max(date.dayofweek - 4, 0))
    return pd.Timestamp(f'{year}-08-15')


In [10]:
def infer_deadlines(history):
    result = {}
    for year in sorted(set(history.index.year)):
        for (degree, fraction) in (('bachelor', 0.2), ('master', 0.25)):
            series = history.loc[str(year), degree]
            threshold = fraction * series.loc[f'{year}-07-01':f'{year}-07-25'].max()
            if threshold <= 0:
                continue
            for (date, count) in series.loc[f'{year}-07-20':f'{year}-08-20'].items():
                following = series.loc[date + pd.Timedelta(days=1):date + pd.Timedelta(days=3)]
                if count >= threshold and len(following) == 3 and (following < threshold).all():
                    result[int(year), degree] = date
                    break
    return result


In [11]:
def history_before(counts, year):
    return counts.loc[(counts.index.year < year) & (counts.index.year != 2020), list(DEGREES)].sort_index().copy()


In [12]:
def categorical_columns():
    return {'weekday': [str(i) for i in range(7)], 'saturday_slot': list(SLOTS), 'deadline_weekday': [str(i) for i in range(7)]}

def features(dates, degree, deadlines=None, coordinates='event', exams=None):
    dates = pd.DatetimeIndex(dates)
    start = pd.to_datetime(dates.year.astype(str) + '-06-20')
    position = (dates - start).days.to_numpy()
    end = pd.DatetimeIndex([(deadlines or {}).get((int(y), degree), nominal_deadline(int(y), degree)) for y in dates.year])
    distance = (dates - end).days.to_numpy()
    work_distance = np.busday_count(end.values.astype('datetime64[D]'), dates.values.astype('datetime64[D]'), weekmask='1111100')
    result = pd.DataFrame({'weekday': dates.dayofweek.astype(str), 'saturday_slot': saturday_slots(dates), 'opening_ramp': np.minimum(position, 14), 'first_day': (position == 0).astype(int), 'distance': distance, 'work_distance': work_distance, 'after_deadline': (distance > 0).astype(int), 'last_day': (distance == 0).astype(int), 'last_workday': ((distance <= 0) & (work_distance == 0)).astype(int), 'last_three_days': ((distance >= -3) & (distance <= 0)).astype(int), 'last_saturday': ((dates.dayofweek == 5) & (distance >= -3) & (distance <= 0)).astype(int), 'deadline_weekday': end.dayofweek.astype(str)}, index=dates)
    if coordinates == 'calendar' or degree == 'background':
        result['position'] = position
        result['day'] = dates.day
        result['month'] = dates.month
        for other in ('bachelor', 'master'):
            other_end = pd.DatetimeIndex([(deadlines or {}).get((int(y), other), nominal_deadline(int(y), other)) for y in dates.year])
            result[f'distance_{other}'] = (dates - other_end).days
    if exams is not None and degree == 'bachelor':
        for subject in exams:
            exam_dates = pd.DatetimeIndex(exams.loc[dates.year, subject])
            result[f'since_exam_{subject}'] = np.clip((dates - exam_dates).days, -14, 35)
    return result


In [13]:
def future_level(levels, year, config):
    if config.level_method == 'none':
        return 0.0
    earlier = levels.loc[levels.index < year].tail(config.level_window)
    (years, values) = (earlier.index.to_numpy(float), earlier.to_numpy(float))
    weight = np.exp2(-(year - years) / config.half_life)
    center = np.average(years, weights=weight)
    mean = np.average(values, weights=weight)
    if config.level_method == 'trend':
        slope = np.sum(weight * (years - center) * (values - mean))
        slope /= np.sum(weight * (years - center) ** 2) + 1.0
        mean += slope * (year - center)
    return max(0.0, float(mean))


In [14]:
def make_design(frame, config, fitted_spline=None):
    if config.family == 'catboost':
        return (frame, None)
    coordinates = ['opening_ramp', 'distance']
    if 'position' in frame:
        coordinates.append('position')
    values = frame[coordinates].to_numpy()
    if fitted_spline is None:
        knots = [np.linspace(0, 14, config.knots), np.linspace(-60, 50, config.knots)]
        if 'position' in frame:
            knots.append(np.linspace(0, 72, config.knots))
        fitted_spline = SplineTransformer(knots=np.array(knots).T, degree=3, include_bias=False, extrapolation='linear').fit(values)
    smooth = fitted_spline.transform(values)
    encoded = frame.copy()
    for (column, categories) in categorical_columns().items():
        encoded[column] = pd.Categorical(encoded[column], categories=categories)
    encoded = pd.get_dummies(encoded, columns=list(categorical_columns()))
    relative = smooth[:, config.knots + 1:2 * (config.knots + 1)]
    weekday = np.column_stack([frame.weekday.eq(str(i)).to_numpy(float) for i in range(1, 6)])
    interaction = (relative[:, :, None] * weekday[:, None, :]).reshape(len(frame), -1)
    matrix = np.column_stack([np.ones((len(frame), 1)), smooth, encoded.to_numpy(float) / 35, interaction])
    return (matrix, fitted_spline)


In [15]:
def train_degree(counts, degree, year, config, exam_dates):
    history = history_before(counts, year)
    if config.history_window:
        preceding = sorted(set(history.index.year))[-config.history_window:]
        history = history.loc[history.index.year.isin(preceding)].copy()
    training_years = sorted(set(map(int, history.index.year)))
    deadlines = infer_deadlines(history) if config.align_history else {}
    exams = None
    if config.exams and degree == 'bachelor':
        exams = exam_dates.loc[exam_dates.index <= year].copy()
    weights = np.exp2(-(year - history.index.year.to_numpy()) / config.half_life)
    slots = saturday_slots(history.index)
    total = history.sum(axis=1)
    closure_rates = {slot: float(np.average(total[slots == slot].eq(0), weights=weights[slots == slot])) for slot in ('first', 'late', 'august', 'regular')}
    target = np.log1p(history[degree])
    weekdays = history.index.dayofweek < 5
    levels = target[weekdays].groupby(history.index.year[weekdays]).mean()
    level = future_level(levels, year, config)
    if config.level_method != 'none':
        target -= history.index.year.map(levels).to_numpy()
    use = history.index.dayofweek != 6
    if config.positive_only:
        use &= history[degree].to_numpy() > 0
    frame = features(history.index, degree, deadlines, config.coordinates, exams)
    (design, spline) = make_design(frame, config)
    fitted = {'degree': degree, 'config': config, 'training_years': training_years, 'exams': exams, 'level': level, 'closure_rates': closure_rates, 'spline': spline}
    if config.family == 'catboost':
        regressor = CatBoostRegressor(iterations=config.iterations, depth=config.depth, learning_rate=config.learning_rate, l2_leaf_reg=config.l2_leaf_reg, loss_function=config.loss, cat_features=list(categorical_columns()), one_hot_max_size=16, random_seed=93001 + DEGREES.index(degree), thread_count=1, allow_const_label=True, verbose=False, allow_writing_files=False)
        regressor.fit(design.loc[use], target.loc[use], sample_weight=weights[use])
        fitted['regressor'] = regressor
    else:
        (matrix, response, weight) = (design[use], target.to_numpy()[use], weights[use])
        gram = np.einsum('ni,n,nj->ij', matrix, weight, matrix)
        penalty = np.full(matrix.shape[1], config.penalty)
        penalty[0] = 1e-08
        gram += np.diag(penalty)
        rhs = np.einsum('ni,n,n->i', matrix, weight, response)
        fitted['coefficients'] = solve(gram, rhs, assume_a='pos')
    return fitted


In [16]:
def predict_degree(fitted, year):
    config = fitted['config']
    degree = fitted['degree']
    dates = campaign(year)
    frame = features(dates, degree, coordinates=config.coordinates, exams=fitted['exams'])
    (design, _) = make_design(frame, config, fitted['spline'])
    if config.family == 'catboost':
        profile = fitted['regressor'].predict(design)
    else:
        profile = np.einsum('ni,i->n', design, fitted['coefficients'])
    log_counts = profile + fitted['level']
    prediction = np.expm1(np.maximum(0.0, log_counts))
    slots = saturday_slots(dates)
    closed = (dates.dayofweek == 6) | np.array([fitted['closure_rates'].get(slot, 0) > 0.5 for slot in slots])
    prediction[closed] = 0.0
    return pd.Series(prediction, index=dates, name=degree)


In [17]:
def random_config(degree, rng):
    common = dict(family='spline' if degree == 'background' else 'catboost', coordinates='calendar' if degree == 'background' else str(rng.choice(['event', 'calendar'])), half_life=float(np.exp(rng.uniform(np.log(0.75), np.log(12.0)))), align_history=bool(rng.integers(2)), level_method=str(rng.choice(['none', 'mean', 'trend'])), level_window=int(rng.choice([2, 3, 5])), history_window=int(rng.choice([0, 3, 5, 7])), positive_only=bool(rng.integers(2)), exams=bool(rng.integers(2)) if degree == 'bachelor' else False)
    if degree == 'background':
        common.update(knots=int(rng.choice([5, 7, 9, 13, 17])), penalty=float(np.exp(rng.uniform(np.log(0.01), np.log(100.0)))))
    else:
        common.update(depth=int(rng.choice([2, 3, 4, 5, 6, 7])), iterations=int(rng.choice([250, 400, 600, 900])), learning_rate=float(np.exp(rng.uniform(np.log(0.015), np.log(0.15)))), l2_leaf_reg=float(np.exp(rng.uniform(np.log(0.3), np.log(80.0)))), loss=str(rng.choice(['RMSE', 'MAE'])))
    return DegreeConfig(**common)

def refine_config(center, rng):
    changes = {'half_life': float(np.clip(center.half_life * np.exp(rng.normal(0, 0.3)), 0.5, 20)), 'level_window': int(rng.choice([2, 3, 5])) if rng.random() < 0.25 else center.level_window, 'history_window': int(rng.choice([0, 3, 5, 7])) if rng.random() < 0.2 else center.history_window, 'level_method': str(rng.choice(['none', 'mean', 'trend'])) if rng.random() < 0.2 else center.level_method, 'align_history': not center.align_history if rng.random() < 0.15 else center.align_history, 'positive_only': not center.positive_only if rng.random() < 0.15 else center.positive_only}
    if center.family == 'catboost':
        changes.update(depth=int(np.clip(center.depth + rng.choice([-1, 0, 1]), 2, 7)), iterations=int(np.clip(center.iterations + rng.choice([-150, 0, 150]), 150, 1200)), learning_rate=float(np.clip(center.learning_rate * np.exp(rng.normal(0, 0.25)), 0.01, 0.18)), l2_leaf_reg=float(np.clip(center.l2_leaf_reg * np.exp(rng.normal(0, 0.6)), 0.1, 150)))
    else:
        changes.update(knots=int(np.clip(center.knots + rng.choice([-2, -1, 0, 1, 2]), 3, 21)), penalty=float(np.clip(center.penalty * np.exp(rng.normal(0, 0.8)), 0.001, 1000)))
    return replace(center, **changes)
reference = v84_reference.copy()
reference['date'] = pd.to_datetime(reference['date'])
y = reference['forecast'].to_numpy(float)

def match_loss(values):
    return np.mean((np.log1p(np.maximum(values, 0)) - np.log1p(y)) ** 2, axis=-1)


## 4. Подбор параметров V93


In [18]:
rngs = {d: np.random.default_rng(SEARCH_SEED + i) for i, d in enumerate(DEGREES)}
candidate_configs = {d: [] for d in DEGREES}
candidate_predictions = {d: [] for d in DEGREES}

def fit_candidates(degree, proposed):
    known = {json.dumps(asdict(c), sort_keys=True) for c in candidate_configs[degree]}
    for config in proposed:
        key = json.dumps(asdict(config), sort_keys=True)
        if key in known:
            continue
        known.add(key)
        fitted = train_degree(daily_counts, degree, TARGET_YEAR, config, exam_calendar)
        candidate_configs[degree].append(config)
        candidate_predictions[degree].append(predict_degree(fitted, TARGET_YEAR).to_numpy(float))

for degree in DEGREES:
    fit_candidates(degree, [random_config(degree, rngs[degree]) for _ in range(CANDIDATES_PER_STREAM)])


In [19]:
def select_combination(predictions):
    arrays = {d: np.asarray(predictions[d]) for d in DEGREES}
    eligible = {d: np.flatnonzero(np.all(arrays[d][:, y == 0] <= 1e-10, axis=1)) for d in DEGREES}
    rng = np.random.default_rng(SEARCH_SEED + 10000)
    starts = [{d: int(eligible[d][0]) for d in DEGREES}]
    starts += [{d: int(rng.choice(eligible[d])) for d in DEGREES} for _ in range(COMBINATION_STARTS - 1)]
    (best_loss, best_indices) = (np.inf, None)
    for (start, selected) in enumerate(starts):
        for _ in range(12):
            previous = selected.copy()
            for degree in DEGREES:
                other = sum((arrays[d][selected[d]] for d in DEGREES if d != degree))
                losses = match_loss(arrays[degree][eligible[degree]] + other)
                selected[degree] = int(eligible[degree][np.argmin(losses)])
            if selected == previous:
                break
        loss = float(match_loss(sum((arrays[d][selected[d]] for d in DEGREES))))
        if loss < best_loss:
            (best_loss, best_indices) = (loss, selected.copy())
    for _ in range(2):
        previous_loss = best_loss
        for (first, second, third) in (('bachelor', 'master', 'background'), ('bachelor', 'background', 'master'), ('master', 'background', 'bachelor')):
            (left, right) = (eligible[first], eligible[second])
            other = arrays[third][best_indices[third]]
            pair_losses = np.stack([match_loss(arrays[first][i] + arrays[second][right] + other) for i in left])
            beam = min(32, pair_losses.size)
            for position in np.argpartition(pair_losses.ravel(), beam - 1)[:beam]:
                (a, b) = np.unravel_index(position, pair_losses.shape)
                pair = arrays[first][left[a]] + arrays[second][right[b]]
                losses = match_loss(arrays[third][eligible[third]] + pair)
                k = int(np.argmin(losses))
                if float(losses[k]) < best_loss - 1e-12:
                    best_loss = float(losses[k])
                    best_indices = {first: int(left[a]), second: int(right[b]), third: int(eligible[third][k])}
        if best_loss >= previous_loss - 1e-12:
            break
    return best_indices

selected_indices = select_combination(candidate_predictions)
for _ in range(REFINEMENT_ROUNDS):
    centers = {d: candidate_configs[d][selected_indices[d]] for d in DEGREES}
    for degree in DEGREES:
        fit_candidates(degree, [refine_config(centers[degree], rngs[degree]) for _ in range(REFINEMENT_PER_STREAM)])
    selected_indices = select_combination(candidate_predictions)

configs = {d: candidate_configs[d][selected_indices[d]] for d in DEGREES}
display(pd.DataFrame({d: asdict(configs[d]) for d in DEGREES}).rename(columns=DEGREE_NAMES))


,Бакалавриат и специалитет,Магистратура,СПО и остальные программы
family,catboost,catboost,spline
coordinates,calendar,event,calendar
depth,7,6,4
iterations,450,750,700
half_life,1.673243,2.595627,7.150261
align_history,True,True,True
level_method,trend,none,none
level_window,2,2,5
exams,True,False,False
knots,9,9,5


## 5. Обучение выбранных моделей


In [20]:
models = {d: train_degree(daily_counts, d, TARGET_YEAR, configs[d], exam_calendar) for d in DEGREES}
v93_components = pd.concat([predict_degree(models[d], TARGET_YEAR) for d in DEGREES], axis=1).reset_index()


## 6. Подбор множителей потоков


In [21]:
streams = list(DEGREES)
components = v93_components.copy()
X = components[streams].to_numpy(float)
target_volume = float(y.sum())

def multiplier_objective(w):
    values = np.einsum('ni,i->n', X, w)
    error = np.log1p(values) - np.log1p(y)
    return float(np.mean(error ** 2) + RIDGE_TO_ONE * np.sum((w - 1.0) ** 2))
initial = np.full(len(streams), target_volume / X.sum())
result = minimize(multiplier_objective, initial, method='SLSQP', bounds=[(0.0, None)] * len(streams), constraints=[{'type': 'eq', 'fun': lambda w: float(np.sum(np.einsum('ni,i->n', X, w)) - target_volume)}], options={'maxiter': 2000, 'ftol': 1e-12})
stream_weights = result.x.copy()
display(pd.DataFrame({'Поток': [DEGREE_NAMES[d] for d in streams], 'Множитель': stream_weights}))


,Поток,Множитель
0,Бакалавриат и специалитет,0.975278
1,Магистратура,1.129799
2,СПО и остальные программы,0.896917


## 7. Итоговый CSV


In [22]:
prediction = np.einsum("ni,i->n", X, stream_weights)
output = pd.DataFrame({"date": components.date.dt.strftime("%Y-%m-%d"), "forecast": prediction})
csv_path = Path("forecast_2024_v93_v84_weighted.csv")
output.to_csv(csv_path, index=False, float_format="%.6f")
print(csv_path)


forecast_2024_v93_v84_weighted.csv
